# SQL for Data Platforms
## Part 12: Same Query, Every Platform

Every prior module ran against SQLite/DuckDB, deliberately, so the whole series needs zero cloud
accounts. This module is different: it's a **reference comparison** of how the same query looks
across BigQuery, Snowflake, Databricks SQL, Microsoft Fabric (T-SQL/Warehouse), and Amazon
Redshift.

**Accuracy guardrail (same standard as the portfolio's "Learn the Pattern" platform-tables):**
every cell below is capability-level, **not** live-tested against a real account — cloud SQL
dialects change. Verify against current vendor docs before relying on any specific syntax.

## Setup — the ANSI baseline, run for real in DuckDB

In [1]:
import duckdb
ddb = duckdb.connect()
ddb.execute("""
CREATE TABLE orders (id INTEGER, customer_id INTEGER, order_date DATE, amount DECIMAL(10,2))
""")
ddb.execute("""
INSERT INTO orders VALUES
    (1, 1, '2024-01-15', 129.99), (2, 1, '2024-03-02', 44.99),
    (3, 2, '2024-02-20', 299.99), (4, 3, '2024-04-11', 89.99)
""")
ddb.execute("SELECT * FROM orders ORDER BY id").df()

,id,customer_id,order_date,amount
0,1,1,2024-01-15,129.99
1,2,1,2024-03-02,44.99
2,3,2,2024-02-20,299.99
3,4,3,2024-04-11,89.99


## 1. Limiting rows

| Platform | Syntax |
|---|---|
| ANSI SQL / DuckDB / PostgreSQL | `SELECT ... LIMIT 10` |
| BigQuery | `SELECT ... LIMIT 10` (same) |
| Snowflake | `SELECT ... LIMIT 10` (same) |
| Databricks SQL | `SELECT ... LIMIT 10` (same) |
| **SQL Server / Fabric Warehouse** | `SELECT TOP 10 ...` (before the column list, not after the query) |
| Redshift (Postgres-derived) | `SELECT ... LIMIT 10` (same) |

Fabric/SQL Server is the outlier here — `TOP` is positional syntax, not a trailing clause, and
(without `WITH TIES`) doesn't guarantee a stable result without an `ORDER BY`.

In [2]:
ddb.execute("SELECT * FROM orders ORDER BY order_date LIMIT 2").df()

,id,customer_id,order_date,amount
0,1,1,2024-01-15,129.99
1,3,2,2024-02-20,299.99


## 2. Date functions

| Platform | Truncate to month | Current date |
|---|---|---|
| ANSI / DuckDB | `DATE_TRUNC('month', order_date)` | `CURRENT_DATE` |
| BigQuery | `DATE_TRUNC(order_date, MONTH)` (arg order flipped) | `CURRENT_DATE()` |
| Snowflake | `DATE_TRUNC('month', order_date)` | `CURRENT_DATE()` |
| Databricks SQL | `DATE_TRUNC('MONTH', order_date)` | `CURRENT_DATE()` |
| Fabric / SQL Server | `DATETRUNC(month, order_date)` (T-SQL, 2022+) or `DATEADD(month, DATEDIFF(month, 0, order_date), 0)` on older versions | `GETDATE()` |
| Redshift | `DATE_TRUNC('month', order_date)` | `CURRENT_DATE` |

BigQuery's `DATE_TRUNC(date_expr, part)` reverses the argument order vs. everyone else's
`DATE_TRUNC(part, date_expr)` — the single most common copy-paste bug moving a query between
BigQuery and any other platform on this list.

In [3]:
ddb.execute("SELECT DATE_TRUNC('month', order_date) AS month, SUM(amount) FROM orders GROUP BY month").df()

,month,sum(amount)
0,2024-01-01,129.99
1,2024-03-01,44.99
2,2024-04-01,89.99
3,2024-02-01,299.99


## 3. String concatenation

| Platform | Syntax |
|---|---|
| ANSI / DuckDB / PostgreSQL / Redshift | `a || b` |
| BigQuery | `CONCAT(a, b)` (`||` also works, but `CONCAT` is the documented idiom) |
| Snowflake | `a || b` or `CONCAT(a, b)` (both supported) |
| Databricks SQL | `CONCAT(a, b)` (`||` also supported) |
| Fabric / SQL Server | `a + b` (T-SQL overloads the arithmetic `+` operator for strings) |

SQL Server/Fabric's `+` is the one that bites people — it silently does *numeric* addition if
either side isn't already a string, instead of raising an error, which BigQuery/Snowflake's
`CONCAT()` won't let happen by accident.

## 4. Semi-structured data (ties to Part 6)

| Platform | Nested/repeated type | Extract a field |
|---|---|---|
| DuckDB / SQLite (this series) | `STRUCT`, `LIST`, JSON functions | `json_extract(col, '$.field')` |
| BigQuery | `STRUCT`, `ARRAY` (native, first-class columns) | `col.field` (dot notation on a STRUCT) |
| Snowflake | `VARIANT`, `OBJECT`, `ARRAY` | `col:field` (colon notation) |
| Databricks SQL | `STRUCT`, `ARRAY`, `MAP` | `col.field` |
| Fabric Warehouse | `JSON` functions (`JSON_VALUE`, `OPENJSON`) over `NVARCHAR` — no first-class nested type yet | `JSON_VALUE(col, '$.field')` |
| Redshift | `SUPER` type | `col.field` |

Unnesting an array (Part 6's `json_each`) also has a different name everywhere: BigQuery
`UNNEST()`, Snowflake `LATERAL FLATTEN`, Databricks `EXPLODE()`, Redshift also `UNNEST` — same
concept, four names.

## 5. Partitioning & clustering (ties to Part 9)

| Platform | Partitioning | Clustering / sort |
|---|---|---|
| BigQuery | `PARTITION BY DATE(col)` on table DDL | `CLUSTER BY col1, col2` |
| Snowflake | Automatic micro-partitioning (no manual partition DDL) | `CLUSTER BY (col)` — a clustering *key*, not a physical sort guarantee |
| Databricks (Delta Lake) | `PARTITIONED BY (col)` | Z-ORDER BY (`OPTIMIZE ... ZORDER BY (col)`), or newer Liquid Clustering |
| Fabric (Warehouse/Lakehouse) | Delta Lake partitioning under the hood (same as Databricks) | evolving — verify current docs |
| Redshift | Distribution style (`DISTKEY`) — different concept, spreads rows across compute nodes | `SORTKEY` — physical sort order on disk |

**Redshift is architecturally different here**, not just syntactically: `DISTKEY` controls *which
node* a row lives on (a distributed-systems concern), while `SORTKEY` controls physical order
*within* a node — the other four platforms separate storage from compute, so they don't have a
"which node" decision to make in the same way (see the portfolio's
[Storage vs. Compute](../../mfzamudio.github.io/publications/pattern-storage-vs-compute.html)).

## 6. `MERGE` support (ties to Part 8)

| Platform | `MERGE` support |
|---|---|
| BigQuery | Yes, full ANSI-style `MERGE` |
| Snowflake | Yes, full ANSI-style `MERGE` |
| Databricks SQL (Delta) | Yes, `MERGE INTO ... USING ... WHEN MATCHED/NOT MATCHED` |
| Fabric Warehouse | Yes (T-SQL `MERGE`) — verify current scope vs. on-prem SQL Server |
| Redshift | No native `MERGE` as of general availability at time of writing — use a staging-table + `DELETE`+`INSERT` pattern instead; **verify current docs**, this is exactly the kind of fast-moving capability this guardrail exists for |

This is the one row on this page most likely to be stale by the time you read it — cloud
warehouses add `MERGE`-family features often.

## Best Practices — Cross-Platform SQL

- Treat every row on this page as "verify before you ship," not as a fact to memorize — these
  platforms ship changes continuously.
- When migrating a query between platforms, check date functions and semi-structured access first
  — they're the two categories with the least syntactic overlap.
- `DISTKEY`/`SORTKEY` (Redshift) are not just "different names" for partitioning/clustering — they
  encode a genuinely different storage architecture. Don't port a partitioning strategy 1:1 without
  understanding that difference.

## Next

**Part 13 — Choosing Your Database** zooms out from syntax to the landscape question: which of
these platforms (plus the OLTP engines and SQLite/DuckDB) actually fits a given problem.